# Medical RAG Bot - Evaluation with LangSmith

We test the chatbot the same way you would test any software: a fixed set of questions
with known answers, run through the bot, scored automatically, with results saved in LangSmith.

**The workflow**
1. **Dataset**: questions taken from our own PDFs, each with a reference answer and the PDF it comes from
2. **Target**: the RAG chain we want to test (the same one `app.py` runs)
3. **Evaluators**: functions that give each answer a score
4. **Experiment**: `client.evaluate()` runs every question through the target, scores it, and uploads the results

**Everything here is free.** The chatbot uses Gemini (Google's free API) and the "LLM-as-a-judge" uses
gpt-oss-120b (Groq's free API). Because the judge is a different model from the chatbot, it is not grading
its own answers. We also use two **code-based** evaluators that don't need an LLM at all.
Read the judge's comments in LangSmith before you trust a score.

> Groq's free tier allows only a limited number of tokens per minute. One run is about 65 API calls
> (13 answers + 52 judge calls), so it's normal for it to pause and retry. Let it finish.

## 1. Setup

In [1]:
import os

if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")

from dotenv import load_dotenv
load_dotenv()

for key in ("PINECONE_API_KEY", "LANGSMITH_API_KEY", "GROQ_API_KEY", "GROQ_MODEL", "GEMINI_API_KEY", "GEMINI_MODEL"):
    assert os.getenv(key), f"{key} missing in .env"

GROQ_MODEL = os.getenv("GROQ_MODEL")
GEMINI_MODEL = os.getenv("GEMINI_MODEL")
print("Working directory:", os.getcwd())
print("Chatbot LLM (Gemini):", GEMINI_MODEL)
print("Judge LLM (Groq):", GROQ_MODEL)

Working directory: d:\Deployed\RAG-Based-Medical-ChatBot
Chatbot LLM (Gemini): gemini-3.5-flash-lite
Judge LLM (Groq): openai/gpt-oss-120b


## 2. Load the chatbot (the *target* we are testing)

In [2]:
from src.helper import download_hugging_face_embeddings, load_llm
from src.rag import load_vector_store, build_rag_chain

embeddings = download_hugging_face_embeddings()
vector_store = load_vector_store(embeddings)
llm = load_llm()   # chatbot LLM: Gemini (GEMINI_MODEL in .env)

K = 3   # number of chunks the retriever returns
rag_chain = build_rag_chain(vector_store, llm, k=K)

In [3]:
def file_name(path):
    """'data\\Diabetes.pdf' (Windows) or 'data/Diabetes.pdf' (Linux) -> 'Diabetes.pdf'"""
    return path.replace("\\", "/").split("/")[-1]


def target(inputs: dict) -> dict:
    """LangSmith calls this once per question in the dataset."""
    result = rag_chain.invoke({"input": inputs["question"]})
    docs = result["context"]
    return {
        "answer": result["answer"].strip(),
        "contexts": [doc.page_content for doc in docs],                    # text of the retrieved chunks
        "sources": [file_name(doc.metadata.get("source", "")) for doc in docs],  # which PDF each came from
    }


# Try it once before running the whole evaluation
test = target({"question": "What is Charcot's triad?"})
print("Answer :", test["answer"])
print("Sources:", test["sources"])

Answer : Charcot's triad is the classical combination of jaundice, upper abdominal pain, and fever. It is associated with acute cholangitis, which occurs when duct obstruction leads to infection and raised intrabiliary pressure.
Sources: ['First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf']


## 3. Create the dataset for groundness check

In [4]:
from langsmith import Client

client = Client()
DATASET_NAME = "medical-bot-eval-v1"

OUT_OF_SCOPE = "I don't know. The medical documents do not contain this information."

examples = [
    # ---------- Diabetes.pdf ----------
    {
        "inputs": {"question": "What is diabetes?"},
        "outputs": {
            "answer": "Diabetes is a chronic health problem characterized by high blood glucose levels "
                      "resulting from defects in insulin production, insulin action, or both.",
            "source": "Diabetes.pdf",
        },
    },
    {
        "inputs": {"question": "How many people in the United States had type 2 diabetes in 2007?"},
        "outputs": {
            "answer": "In 2007, 23.6 million people, or 7.8% of the United States population, had type 2 diabetes.",
            "source": "Diabetes.pdf",
        },
    },
    # ---------- Allergen Immunotherapy.pdf ----------
    {
        "inputs": {"question": "What color codes are used for allergen immunotherapy vials?"},
        "outputs": {
            "answer": "Red is the maintenance concentrate (1:1 vol/vol, #1), yellow is the 10-fold dilution "
                      "(1:10, #2), blue is the 100-fold dilution (1:100, #3), green is the 1000-fold dilution "
                      "(1:1000, #4) and silver is the 10,000-fold dilution (1:10,000, #5).",
            "source": "Allergen Immunotherapy.pdf",
        },
    },
    {
        "inputs": {"question": "How long is the expiration dating period for 50% glycerin bulk allergen extracts?"},
        "outputs": {
            "answer": "A total of six years from the time of extraction: a maximum of three years for "
                      "manufacturer storage and three years for shipped vial expiration dating.",
            "source": "Allergen Immunotherapy.pdf",
        },
    },
    # ---------- STANDARD TREATMENT.pdf ----------
    {
        "inputs": {"question": "How should P. vivax malaria be treated?"},
        "outputs": {
            "answer": "With chloroquine for three days and primaquine for 14 days. Primaquine prevents relapse "
                      "but is contraindicated in pregnant women, infants and people with G6PD deficiency.",
            "source": "STANDARD TREATMENT.pdf",
        },
    },
    {
        "inputs": {"question": "What are the types of poisonous snake bite?"},
        "outputs": {
            "answer": "Poisonous snake bites are of two types: neurotoxic and vasculotoxic.",
            "source": "STANDARD TREATMENT.pdf",
        },
    },
    # ---------- First Principles of Gastroenterology.pdf ----------
    {
        "inputs": {"question": "What is Charcot's triad?"},
        "outputs": {
            "answer": "Intermittent fever and chills, jaundice and abdominal pain. It is the classical "
                      "presentation of acute bacterial (ascending) cholangitis.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    {
        "inputs": {"question": "How is H. pylori infection treated?"},
        "outputs": {
            "answer": "With triple therapy: a proton pump inhibitor (PPI) plus two appropriate antibiotics taken "
                      "twice daily for 7-14 days. A urea breath test after the patient has been off the PPI "
                      "for 2 weeks confirms eradication.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    {
        "inputs": {"question": "What does a high serum-ascites albumin gradient (SAAG) indicate?"},
        "outputs": {
            "answer": "A SAAG above 11 g/L means the ascites is due to portal hypertension (for example cirrhosis). "
                      "A low gradient means it is not from portal hypertension; malignancy is the likeliest cause.",
            "source": "First Principles of Gastroenterology.pdf",
        },
    },
    # ---------- clinical_medicine_ashok_chandra.pdf ----------
    {
        "inputs": {"question": "What does a complete clinical diagnosis include?"},
        "outputs": {
            "answer": "The site of disease (anatomical), the nature of disease (etiological), the "
                      "pathophysiology (complications) and the functional disturbance (dysfunction).",
            "source": "clinical_medicine_ashok_chandra.pdf",
        },
    },
    {
        "inputs": {"question": "What are the common causes of cardiac disease?"},
        "outputs": {
            "answer": "Congenital, rheumatic, hypertensive, infectious (e.g. tuberculous or viral pericarditis, "
                      "viral myocarditis, subacute bacterial endocarditis) and atherosclerotic heart disease.",
            "source": "clinical_medicine_ashok_chandra.pdf",
        },
    },
    # ---------- Out of scope: the bot should say it doesn't know ----------
    {
        "inputs": {"question": "Who won the 2011 Cricket World Cup?"},
        "outputs": {"answer": OUT_OF_SCOPE, "source": None},
    },
    {
        "inputs": {"question": "How do I reset my Wi-Fi router?"},
        "outputs": {"answer": OUT_OF_SCOPE, "source": None},
    },
]

if client.has_dataset(dataset_name=DATASET_NAME):
    print(f"Dataset '{DATASET_NAME}' already exists - reusing it.")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Questions from the 5 medical PDFs + 2 out-of-scope questions",
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset '{DATASET_NAME}' with {len(examples)} examples.")

Dataset 'medical-bot-eval-v1' already exists - reusing it.


## 4. Evaluators

An evaluator is a function that receives the question (`inputs`), the bot's output (`outputs`) and,
if it needs one, the reference (`reference_outputs`), and returns a score.
LangSmith works out which arguments to pass from the **parameter names**, so keep these names exactly.

Each evaluator returns `{"key": name, "score": number, "comment": why}`. The comment shows up in LangSmith.

| Evaluator | Compares | Uses an LLM? | Question it answers |
|---|---|---|---|
| `source_hit` | retrieved sources vs reference source | No | Did retrieval find the right PDF? |
| `answer_similarity` | answer vs reference answer | No (embeddings) | How close in meaning is the answer? (0-1) |
| `correctness` | answer vs reference answer | Yes | Is the answer factually right? |
| `groundedness` | answer vs retrieved chunks | Yes | Did the bot stick to the documents, or make things up? |
| `relevance` | answer vs question | Yes | Does the answer address the question? |
| `retrieval_relevance` | retrieved chunks vs question | Yes | Were the retrieved chunks about the question? |

### 4.1 Code-based evaluators (no LLM, so they are fast and give the same result every run)

In [5]:
import numpy as np


def source_hit(outputs: dict, reference_outputs: dict) -> dict:
    """1 if any retrieved chunk came from the PDF that contains the answer, else 0."""
    expected = reference_outputs["source"]
    if expected is None:   # out-of-scope question: there is no correct PDF
        return {"key": "source_hit", "score": None, "comment": "Out-of-scope question, not scored"}

    hit = expected in outputs["sources"]
    return {
        "key": "source_hit",
        "score": int(hit),
        "comment": f"expected {expected} | retrieved {outputs['sources']}",
    }


def answer_similarity(outputs: dict, reference_outputs: dict) -> dict:
    """Cosine similarity between the embeddings of the bot answer and the reference answer."""
    a = np.array(embeddings.embed_query(outputs["answer"]))
    b = np.array(embeddings.embed_query(reference_outputs["answer"]))
    cosine = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    return {"key": "answer_similarity", "score": round(cosine, 3)}

### 4.2 LLM-as-a-judge evaluators

We send the judge clear instructions and ask it to **start its reply with YES or NO**, then give one
sentence of reasoning. `YES` becomes score 1 and `NO` becomes score 0. The full reply is kept as the comment, so
you can see *why* it decided.

In [6]:
from langchain_groq import ChatGroq
from langchain_core.rate_limiters import InMemoryRateLimiter

# Groq free tier: 8,000 tokens per minute. One judge call uses ~700-1,000 tokens,
# so allow at most one call every 8 seconds (about 7-8 calls per minute).
judge_rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.125,   # 1 call every 8 seconds
    check_every_n_seconds=0.5,
    max_bucket_size=1,           # no bursts
)

# Judge = gpt-oss-120b on Groq, a different model from the Gemini chatbot
judge_llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,               # same answer -> same grade
    reasoning_effort="low",      # shorter hidden reasoning = fewer tokens per call
    max_retries=5,               # wait and retry when the free-tier rate limit is hit
    rate_limiter=judge_rate_limiter,
)


def ask_judge(instructions: str, content: str):
    """Send one grading request to the judge and turn its YES/NO reply into a 1/0 score."""
    reply = judge_llm.invoke([("system", instructions), ("human", content)]).content.strip()
    first_word = reply.lstrip("*#> ").upper()
    score = 1 if first_word.startswith("YES") else 0
    return score, reply


def join_chunks(contexts: list) -> str:
    return "\n\n".join(f"[Chunk {i + 1}] {text}" for i, text in enumerate(contexts))

In [7]:
CORRECTNESS_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION, the CORRECT ANSWER and a STUDENT ANSWER.
Grade the STUDENT ANSWER only on whether its facts agree with the CORRECT ANSWER.
Extra information is fine as long as it does not contradict the CORRECT ANSWER.
If the CORRECT ANSWER says "I don't know", the STUDENT ANSWER is correct only if it also says it doesn't know.

Start your reply with YES (correct) or NO (incorrect), then give one short sentence explaining why."""


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    content = (
        f"QUESTION: {inputs['question']}\n"
        f"CORRECT ANSWER: {reference_outputs['answer']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )
    score, reply = ask_judge(CORRECTNESS_INSTRUCTIONS, content)
    return {"key": "correctness", "score": score, "comment": reply}

In [8]:
GROUNDEDNESS_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given FACTS and a STUDENT ANSWER.
The STUDENT ANSWER is grounded if everything it says is supported by the FACTS.
It is NOT grounded if it contains information that is not in the FACTS (made-up information).
An answer that only says "I don't know" is grounded.

Start your reply with YES (grounded) or NO (not grounded), then give one short sentence explaining why."""


def groundedness(outputs: dict) -> dict:
    content = f"FACTS:\n{join_chunks(outputs['contexts'])}\n\nSTUDENT ANSWER: {outputs['answer']}"
    score, reply = ask_judge(GROUNDEDNESS_INSTRUCTIONS, content)
    return {"key": "groundedness", "score": score, "comment": reply}

In [9]:
RELEVANCE_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION and a STUDENT ANSWER.
The STUDENT ANSWER is relevant if it is concise and actually helps answer the QUESTION.

Start your reply with YES (relevant) or NO (not relevant), then give one short sentence explaining why."""


def relevance(inputs: dict, outputs: dict) -> dict:
    content = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    score, reply = ask_judge(RELEVANCE_INSTRUCTIONS, content)
    return {"key": "relevance", "score": score, "comment": reply}

In [10]:
RETRIEVAL_RELEVANCE_INSTRUCTIONS = """You are a teacher grading a quiz.
You will be given a QUESTION and some FACTS that were retrieved to answer it.
The FACTS are relevant if they contain any keywords or meaning related to the QUESTION.
They are NOT relevant if they are completely unrelated to the QUESTION.

Start your reply with YES (relevant) or NO (not relevant), then give one short sentence explaining why."""


def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    content = f"QUESTION: {inputs['question']}\n\nFACTS:\n{join_chunks(outputs['contexts'])}"
    score, reply = ask_judge(RETRIEVAL_RELEVANCE_INSTRUCTIONS, content)
    return {"key": "retrieval_relevance", "score": score, "comment": reply}

### 4.3 Test the evaluators on one answer before running everything
It's cheaper to catch a bug here than halfway through the experiment.

In [11]:
sample = examples[6]
sample_out = target(sample["inputs"])

for evaluator in [source_hit, answer_similarity]:
    print(evaluator(outputs=sample_out, reference_outputs=sample["outputs"]))
print(correctness(sample["inputs"], sample_out, sample["outputs"]))
print(groundedness(sample_out))
print(relevance(sample["inputs"], sample_out))
print(retrieval_relevance(sample["inputs"], sample_out))

{'key': 'source_hit', 'score': 1, 'comment': "expected First Principles of Gastroenterology.pdf | retrieved ['First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf', 'First Principles of Gastroenterology.pdf']"}
{'key': 'answer_similarity', 'score': 0.589}
{'key': 'correctness', 'score': 1, 'comment': "YES. The student correctly lists jaundice, abdominal pain, and fever as Charcot's triad, matching the correct answer."}
{'key': 'groundedness', 'score': 1, 'comment': 'YES. The answer correctly states Charcot’s triad and its link to acute cholangitis, which are both described in the provided facts.'}
{'key': 'relevance', 'score': 1, 'comment': "YES – the answer correctly and concisely defines Charcot's triad as jaundice, upper abdominal (right‑upper‑quadrant) pain, and fever, which is exactly what the question asks."}
{'key': 'retrieval_relevance', 'score': 1, 'comment': 'YES – Chunk\u202f1 directly defines Charcot’s triad as jaundice, upper abdominal pain,

### 4.4 Context precision and context recall

These two check the **retrieval step** in more detail than `source_hit`. Both compare the retrieved
chunks with the **correct answer** from the dataset.

| Metric | Question it answers | Example with k = 3 |
|---|---|---|
| `context_precision` | Of the chunks we retrieved, how many were actually **useful**, and were the useful ones **ranked first**? | Useful pattern `[1, 0, 0]` → 1.0, `[0, 0, 1]` → 0.33 |
| `context_recall` | Of the facts in the correct answer, how many were **present** in the retrieved chunks? | The answer has 3 facts and the chunks contain 2 → 0.67 |

- **Low precision**: the retriever brings back noise (junk chunks crowd the prompt).
- **Low recall**: the retriever **misses** facts the answer needs, so the LLM *cannot* answer fully,
  however good it is.

**How the precision score is calculated (average precision).** The judge marks each chunk as useful (1) or not (0).
At every useful chunk we compute *precision so far* = useful chunks up to this rank ÷ rank, then average those values.
So a useful chunk at rank 1 counts more than one at rank 3: `[1, 0, 1]` → (1/1 + 2/3) ÷ 2 = 0.83.

Each metric needs **one** judge call per question: the judge gives all its verdicts at once, one per line.
Out-of-scope questions are not scored, because there is no correct answer to look for in the chunks.

In [12]:
import re

CONTEXT_PRECISION_INSTRUCTIONS = """You are a teacher checking a student's research notes.
You will be given a QUESTION, the CORRECT ANSWER and numbered CHUNKS of text that were retrieved to answer it.
For EACH chunk, decide whether it contains information that helps produce the CORRECT ANSWER.

Reply with exactly one line per chunk, in order, in this format and nothing else:
Chunk 1: YES
Chunk 2: NO"""


def context_precision(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Of the retrieved chunks, how many are useful - with useful chunks at the top counting more."""
    if reference_outputs["source"] is None:   # out-of-scope question: no correct answer to look for
        return {"key": "context_precision", "score": None, "comment": "Out-of-scope question, not scored"}

    content = (
        f"QUESTION: {inputs['question']}\n"
        f"CORRECT ANSWER: {reference_outputs['answer']}\n\n"
        f"CHUNKS:\n{join_chunks(outputs['contexts'])}"
    )
    reply = judge_llm.invoke([("system", CONTEXT_PRECISION_INSTRUCTIONS), ("human", content)]).content.strip()

    # Read "Chunk 1: YES" lines -> {1: True, 2: False, ...}; a chunk the judge skipped counts as not useful
    verdicts = {int(n): v.upper() == "YES"
                for n, v in re.findall(r"Chunk\s*(\d+)\s*\**\s*:\s*\**\s*(YES|NO)", reply, flags=re.IGNORECASE)}
    useful = [1 if verdicts.get(rank, False) else 0 for rank in range(1, len(outputs["contexts"]) + 1)]

    # Average precision: at each useful chunk, precision so far = useful chunks up to here / rank
    hits, precisions = 0, []
    for rank, is_useful in enumerate(useful, start=1):
        if is_useful:
            hits += 1
            precisions.append(hits / rank)
    score = sum(precisions) / len(precisions) if precisions else 0.0

    return {"key": "context_precision", "score": round(score, 3),
            "comment": f"useful chunks by rank: {useful}\n{reply}"}

In [13]:
CONTEXT_RECALL_INSTRUCTIONS = """You are a teacher checking whether a student's notes cover the correct answer.
You will be given the CORRECT ANSWER and some CHUNKS of retrieved text.
Split the CORRECT ANSWER into its separate facts (usually 1 to 5).
For each fact, decide whether the CHUNKS contain that fact.

Reply with exactly one line per fact, in this format and nothing else:
FACT: <the fact> -> YES
FACT: <the fact> -> NO"""


def context_recall(outputs: dict, reference_outputs: dict) -> dict:
    """What fraction of the facts in the correct answer were present in the retrieved chunks?"""
    if reference_outputs["source"] is None:   # out-of-scope question: no correct answer to look for
        return {"key": "context_recall", "score": None, "comment": "Out-of-scope question, not scored"}

    content = (
        f"CORRECT ANSWER: {reference_outputs['answer']}\n\n"
        f"CHUNKS:\n{join_chunks(outputs['contexts'])}"
    )
    reply = judge_llm.invoke([("system", CONTEXT_RECALL_INSTRUCTIONS), ("human", content)]).content.strip()

    # Read the verdict at the end of each "FACT: ... -> YES/NO" line
    verdicts = re.findall(r"->\s*\**\s*(YES|NO)", reply, flags=re.IGNORECASE)
    if not verdicts:
        return {"key": "context_recall", "score": None, "comment": f"Could not read the judge's reply:\n{reply}"}

    found = sum(v.upper() == "YES" for v in verdicts)
    return {"key": "context_recall", "score": round(found / len(verdicts), 3),
            "comment": f"{found} of {len(verdicts)} facts found in the chunks\n{reply}"}

In [14]:
# Test both on the same sample answer as section 4.3
print(context_precision(sample["inputs"], sample_out, sample["outputs"]))
print(context_recall(sample_out, sample["outputs"]))

{'key': 'context_precision', 'score': 1.0, 'comment': 'useful chunks by rank: [1, 0, 0]\nChunk 1: YES\nChunk 2: NO\nChunk 3: NO'}
{'key': 'context_recall', 'score': 1.0, 'comment': '4 of 4 facts found in the chunks\nFACT: Intermittent fever and chills -> YES\nFACT: Jaundice -> YES\nFACT: Abdominal pain -> YES\nFACT: It is the classical presentation of acute bacterial (ascending) cholangitis -> YES'}


### 4.5 Hallucination rate

`groundedness` (section 4.2) only says YES or NO for a whole answer. This metric measures **how much** of the answer
is made up:

1. The judge splits the bot's answer into its separate claims (single facts).
2. For each claim it checks whether the retrieved chunks support it.
3. **Hallucination rate = unsupported claims ÷ total claims.** Shown as 0-1 in LangSmith; section 6 prints it as a percentage.

**Lower is better** (0% = every claim comes from the documents), unlike the other metrics.

Example: the answer *"Chloroquine for 3 days, primaquine for 14 days, and paracetamol for fever"* has 3 claims.
If the chunks mention only the first two, the rate is 1 ÷ 3 = 33%, even though paracetamol is reasonable medicine,
because it did not come from the documents.

Out-of-scope questions **are** scored here: a bot that says "I don't know" makes no claims (0%), while a bot that
answers "India won the 2011 World Cup" from its own knowledge makes an unsupported claim (100%).

In [15]:
HALLUCINATION_INSTRUCTIONS = """You are a fact-checker.
You will be given FACTS (retrieved text) and a STUDENT ANSWER.
Split the STUDENT ANSWER into its separate factual claims.
For each claim, decide whether the FACTS support it.

Reply with exactly one line per claim, in this format and nothing else:
CLAIM: <the claim> -> SUPPORTED
CLAIM: <the claim> -> NOT SUPPORTED

If the STUDENT ANSWER makes no factual claims (for example it only says it does not know), reply exactly:
NO CLAIMS"""


def hallucination_rate(outputs: dict) -> dict:
    """Share of the answer's claims that are NOT supported by the retrieved chunks (0 = none made up)."""
    content = f"FACTS:\n{join_chunks(outputs['contexts'])}\n\nSTUDENT ANSWER: {outputs['answer']}"
    reply = judge_llm.invoke([("system", HALLUCINATION_INSTRUCTIONS), ("human", content)]).content.strip()

    if reply.lstrip("*# ").upper().startswith("NO CLAIMS"):
        return {"key": "hallucination_rate", "score": 0.0, "comment": "No factual claims (e.g. 'I don't know')"}

    # Read the verdict at the end of each "CLAIM: ... -> SUPPORTED / NOT SUPPORTED" line
    verdicts = re.findall(r"->\s*\**\s*(NOT SUPPORTED|SUPPORTED)", reply, flags=re.IGNORECASE)
    if not verdicts:
        return {"key": "hallucination_rate", "score": None, "comment": f"Could not read the judge's reply:\n{reply}"}

    unsupported = sum(v.upper() == "NOT SUPPORTED" for v in verdicts)
    return {"key": "hallucination_rate", "score": round(unsupported / len(verdicts), 3),
            "comment": f"{unsupported} of {len(verdicts)} claims not supported by the chunks\n{reply}"}

In [16]:
# Test on the same sample answer as section 4.3
print(hallucination_rate(sample_out))

{'key': 'hallucination_rate', 'score': 0.0, 'comment': "0 of 3 claims not supported by the chunks\nCLAIM: Charcot's triad is the classical combination of jaundice, upper abdominal pain, and fever. -> SUPPORTED\nCLAIM: It is associated with acute cholangitis. -> SUPPORTED\nCLAIM: Acute cholangitis occurs when duct obstruction leads to infection. -> SUPPORTED"}


## 5. Run the evaluation (baseline experiment)

In [17]:
EVALUATORS = [source_hit, answer_similarity, correctness, groundedness, relevance, retrieval_relevance,
              context_precision, context_recall, hallucination_rate]

baseline = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix=f"gemini-bot-gptoss-judge-k{K}",
    metadata={"llm": GEMINI_MODEL, "judge": GROQ_MODEL, "embeddings": "all-MiniLM-L6-v2", "k": K},
    max_concurrency=1,
)

View the evaluation results for experiment: 'gemini-bot-gptoss-judge-k3-a99baed3' at:
https://smith.langchain.com/o/8b966b74-a972-4bc3-a3db-0c647723e993/datasets/c846f119-a3c6-4748-979d-06a5941bce8c/compare?selectedSessions=793de172-dd8d-48c7-965d-513d598f22f3




0it [00:00, ?it/s]

## 6. Look at the results

In [18]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

df = baseline.to_pandas()

score_columns = [c for c in df.columns if c.startswith("feedback.")]
df[["inputs.question", "outputs.answer"] + score_columns]

,inputs.question,outputs.answer,feedback.source_hit,feedback.answer_similarity,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,feedback.context_precision,feedback.context_recall,feedback.hallucination_rate
0,How is H. pylori infection treated?,"Based on the provided documents, the treatment",1.0,0.349,0,1,0,1,0.000,0.0,0.000
1,How many people in the United States had type 2 diabetes in 2007?,I don't know. The medical documents do not contain this information.,1.0,0.222,0,0,0,0,0.000,0.0,0.000
2,What does a complete clinical diagnosis include?,"A complete clinical diagnosis incorporates four parts: the nature, site, pat...",1.0,0.610,1,1,1,1,0.500,1.0,0.000
3,What are the common causes of cardiac disease?,"Based on the provided documents, the common etiologies of cardiac disease ar...",1.0,0.733,0,1,1,1,1.000,0.8,0.000
4,How should P. vivax malaria be treated?,*P. vivax* cases should be treated with chloroquine for three days and Prima...,1.0,0.642,1,1,1,1,0.583,1.0,0.000
5,What are the types of poisonous snake bite?,"Based on the provided documents, poisonous snake bites are of two types: neu...",1.0,0.933,1,1,1,1,1.000,1.0,0.000
6,What color codes are used for allergen immunotherapy vials?,"Based on the provided documents, the color codes used for allergen immunothe...",1.0,0.677,0,1,1,1,0.500,0.8,0.000
7,How long is the expiration dating period for 50% glycerin bulk allergen extr...,A total of six years from the time of extraction is allotted to 50% glycerin...,1.0,0.826,1,1,1,1,0.500,1.0,0.333
8,What is diabetes?,Diabetes is a chronic health problem characterized by high blood glucose lev...,1.0,0.980,1,1,1,1,0.500,1.0,0.000
9,How do I reset my Wi-Fi router?,I don't know. The medical documents do not contain this information.,NaN,1.000,1,1,0,0,NaN,NaN,0.000


In [19]:
df[score_columns].mean().round(2)

feedback.source_hit             1.00
feedback.answer_similarity      0.72
feedback.correctness            0.69
feedback.groundedness           0.92
feedback.relevance              0.69
feedback.retrieval_relevance    0.77
feedback.context_precision      0.55
feedback.context_recall         0.74
feedback.hallucination_rate     0.03
dtype: float64

In [20]:
# Hallucination as percentages (lower is better)
hallucination = df["feedback.hallucination_rate"].dropna()
print(f"Average hallucination rate             : {hallucination.mean() * 100:.1f}% of claims not supported by the documents")
print(f"Answers with at least one hallucination: {(hallucination > 0).mean() * 100:.1f}% "
      f"({(hallucination > 0).sum()} of {len(hallucination)} answers)")

Average hallucination rate             : 2.6% of claims not supported by the documents
Answers with at least one hallucination: 7.7% (1 of 13 answers)


In [21]:
df[df["feedback.correctness"] == 0][["inputs.question", "outputs.answer", "reference.answer"]]

,inputs.question,outputs.answer,reference.answer
0,How is H. pylori infection treated?,"Based on the provided documents, the treatment",With triple therapy: a proton pump inhibitor (PPI) plus two appropriate anti...
1,How many people in the United States had type 2 diabetes in 2007?,I don't know. The medical documents do not contain this information.,"In 2007, 23.6 million people, or 7.8% of the United States population, had t..."
3,What are the common causes of cardiac disease?,"Based on the provided documents, the common etiologies of cardiac disease ar...","Congenital, rheumatic, hypertensive, infectious (e.g. tuberculous or viral p..."
6,What color codes are used for allergen immunotherapy vials?,"Based on the provided documents, the color codes used for allergen immunothe...","Red is the maintenance concentrate (1:1 vol/vol, #1), yellow is the 10-fold ..."


**How to read the results.** The metrics point to different parts of the system:
- `source_hit` = 0 → a **retrieval** problem. The right PDF never reached the LLM, so the LLM had no chance.
- `source_hit` = 1 but `correctness` = 0 → a **generation** problem. The facts were there, but the model didn't use them well.
- `groundedness` = 0 → the model **made something up** that isn't in the chunks.
- Out-of-scope questions with `correctness` = 0 → the bot answered from its own knowledge instead of saying "I don't know".

Open the experiment link printed above to see every question, the retrieved chunks, and each judge's comment.

## 7. Experiment: change one thing and compare

Evaluation is useful because you can **compare**. Change one setting, run again with a new
`experiment_prefix`, then in LangSmith open the dataset → select both experiments → **Compare**.

Here we try retrieving 5 chunks instead of 3. Before running it, predict: which metrics should go up,
and which might go down?

In [22]:
K = 5
rag_chain = build_rag_chain(vector_store, llm, k=K)

experiment_k5 = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix=f"gemini-bot-gptoss-judge-k{K}",
    metadata={"llm": GEMINI_MODEL, "judge": GROQ_MODEL, "embeddings": "all-MiniLM-L6-v2", "k": K},
    max_concurrency=1,
)

comparison = pd.DataFrame({
    "k=3": baseline.to_pandas()[score_columns].mean(),
    "k=5": experiment_k5.to_pandas()[score_columns].mean(),
}).round(2)
comparison

View the evaluation results for experiment: 'gemini-bot-gptoss-judge-k5-84cae8ff' at:
https://smith.langchain.com/o/8b966b74-a972-4bc3-a3db-0c647723e993/datasets/c846f119-a3c6-4748-979d-06a5941bce8c/compare?selectedSessions=e480992f-977e-4767-92d2-04def767c1b9




0it [00:00, ?it/s]

,k=3,k=5
feedback.source_hit,1.00,1.00
feedback.answer_similarity,0.72,0.71
feedback.correctness,0.69,0.62
feedback.groundedness,0.92,1.00
feedback.relevance,0.69,0.69
feedback.retrieval_relevance,0.77,0.77
feedback.context_precision,0.55,0.54
feedback.context_recall,0.74,0.80
feedback.hallucination_rate,0.03,0.04
